# Zero-noise extrapolation

ZNE runs copies of your circuit with more noise than the original, then extrapolates the measured values back to the zero-noise limit.

Noise is amplified by global folding: `U` becomes `U(U†U)^s`, so every gate runs `1 + 2s` times. You choose the noise factors `1 + 2s`, which are odd integers starting at 1.

An observable is a weighted sum of Pauli terms, and ZNE fits each Pauli term on its own.

`ErrorReductionWorkflow` runs in four steps, `plan`, `submit`, `collect` and `estimate`, and each step stores its result on the workflow. The workflow can be saved after any step and reloaded in a later session.

## 0. Setup

In [ ]:
from iqm.error_reduction_tools.observables import ObservableSet
from iqm.error_reduction_tools.readout_characterization.readout_error_characterization import RECConfiguration
from iqm.error_reduction_tools.twirling.twirling_api import TwirlingConfiguration
from iqm.error_reduction_tools.utils.qiskit_utils import canonicalize_circuit
from iqm.error_reduction_tools.workflow import ErrorReductionWorkflow
from iqm.error_reduction_tools.zne import ZNE, ZNEEstimation, plot_zne_comparison, plot_zne_fit
from iqm.error_reduction_tools.zne.zne_plots import COMPARISON_COLOURS
from iqm.qiskit_iqm import IQMProvider
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from iqm.iqm_client.util import print_env_vars

See the IQM Resonance documentation for how to connect. The workflow submits the circuits through the backend's client.

In [ ]:
provider = IQMProvider("https://resonance.iqm.tech", quantum_computer="garnet")
backend = provider.get_backend()

## 1. Circuits and observables

Two 4-qubit GHZ circuits with a small `RX` rotation per qubit, differing in the angle, so neither is Clifford.

We measure `ZZZZ` and `ZZII + 0.75 ZIZI`. That is three Pauli terms, so three fits per circuit.

Pauli labels are little-endian: the rightmost character is classical bit 0.

In [ ]:
NUM_QUBITS = 4
SHOTS = 10_000
OBSERVABLES = ["ZZZZ", [(1.0, "ZZII"), (0.75, "ZIZI")]]
OBSERVABLE_NAMES = ["ZZZZ", "ZZII + 0.75 ZIZI"]

def ghz_with_rotation(num_qubits, angle):
    """GHZ chain plus a small RX rotation per qubit, so the circuit is not Clifford."""
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for qubit in range(num_qubits - 1):
        qc.cx(qubit, qubit + 1)
    for qubit in range(num_qubits):
        qc.rx(angle, qubit)
    return qc


def exact_distribution(qc):
    """Return the noiseless output distribution of an unmeasured circuit. Feasible only because it is small."""
    probabilities = Statevector.from_instruction(qc).probabilities()
    return {format(i, f"0{qc.num_qubits}b"): float(p) for i, p in enumerate(probabilities) if p > 1e-15}


angles = [0.25, 0.50]
bare = [ghz_with_rotation(NUM_QUBITS, angle) for angle in angles]

observables = ObservableSet.build(OBSERVABLES)
ideal_terms = [observables.term_values_from_counts(exact_distribution(qc)) for qc in bare]
ideal_observables = [observables.combine(terms) for terms in ideal_terms]
print("Pauli terms:", observables.pauli_terms)

# The workflow takes iqm.pulse circuits. Transpile to the QPU, then convert, naming qubits after the backend.
targets = []
for angle, qc in zip(angles, bare):
    measured = qc.copy()
    measured.measure_all()
    transpiled = transpile(measured, backend=backend, optimization_level=3)
    pulse_circuit, _ = canonicalize_circuit(transpiled, name=f"ghz_{angle}", backend=backend)
    targets.append(pulse_circuit)

## 2. The workflow

`ZNE` decides which circuits run on the QPU and cannot change once they are submitted. `ZNEEstimation`, given later to `estimate`, is everything you can redo for free once the data exists.

In [ ]:
workflow = ErrorReductionWorkflow(
    targets,
    OBSERVABLES,
    ZNE(noise_factors=(1, 3, 7, 9)),
    client=backend.client,
    twirling_configuration=TwirlingConfiguration(readout_twirl_strategy="LOCAL", num_twirling_instances=20, seed=42),
    rec_configuration=RECConfiguration(),
    shots=SHOTS,
)

**Costs QPU time** (fixed once the circuits are submitted)

| parameter | options | default |
| --- | --- | --- |
| `ZNE.noise_factors` | odd integers in increasing order, starting at `1` | `(1, 3, 5)` |
| `shots` | total shots of each noise factor, split over its twirling instances | `20_000` |
| `twirling_configuration.readout_twirl_strategy` | `"LOCAL"`, `"MINIMAL"`, `"HADAMARD"`, `"NONE"` | `"LOCAL"` |
| `twirling_configuration.circuit_twirling` | `True`, `False` | `True` |
| `twirling_configuration.num_twirling_instances` | any positive `int` | `20` |
| `twirling_configuration` | `None` executes every circuit once, without twirling | |
| `rec_configuration.num_circuits`, `rec_configuration.shots` | size of the readout characterization job; `rec_configuration=None` skips it | `50`, `10_000` |

**Free to redo** (`ZNEEstimation` and the options of `estimate`, no QPU involved)

| parameter | options | default |
| --- | --- | --- |
| `extrapolation_type` | `"Poly"`, `"Exp"` | `"Poly"` |
| `degree` | any `int` up to one less than the number of fitted points, for `"Poly"` only | `None`, i.e. Richardson |
| `noise_factors` | any two or more of the executed noise factors, or `None` for all of them | `None` |
| `apply_rem` | `True`, `False`, or `None` to mitigate whenever readout errors were characterized | `None` |
| `bootstrap_samples` | `0` for no uncertainties, or the number of resamples of the counts | `0` |

## 3. Plan, submit, collect

`plan` is free and shows what will run. `submit` returns once the jobs are queued. `collect` waits for them and keeps the counts, which is the expensive data. Save the workflow to keep them.

In [ ]:
plan = workflow.plan()
print(plan.summary())

In [ ]:
submission = workflow.submit()
print("job ids:", [job.job_id for job in submission.jobs])

In [ ]:
workflow.collect()
workflow.save("zne_workflow.json")

## 4. Estimate

Everything below needs no quantum computer, so a later session starts here. Uncertainties come from resampling the measured counts and repeating the whole estimation, so leave `bootstrap_samples` at `0` while you are still choosing a fit.

In [ ]:
workflow = ErrorReductionWorkflow.load("zne_workflow.json")

BOOTSTRAP_SAMPLES = 200
result = workflow.estimate(ZNEEstimation(degree=1), bootstrap_samples=BOOTSTRAP_SAMPLES)
print("estimate:   ", result.values)
print("unmitigated:", result.unmitigated)
print("ideal:      ", np.asarray(ideal_observables))

One panel per Pauli term, which is what was fitted. `ideal` adds the noiseless reference to each panel.

In [ ]:
fig = plot_zne_fit(result, target_index=1, ideal=ideal_terms[1])
plt.show()

### 4.1 The fitting function

- a polynomial of the given `degree`;
- `degree=None` is Richardson extrapolation, the polynomial through every fitted point;
- `extrapolation_type="Exp"` fits `a * exp(b * x)`.

In [ ]:
fits = {
    "Poly deg 1": ZNEEstimation(degree=1),
    "Poly deg 2": ZNEEstimation(degree=2),
    "Richardson": ZNEEstimation(),
    "Exp": ZNEEstimation(extrapolation_type="Exp"),
}
results = {label: workflow.estimate(fit, bootstrap_samples=BOOTSTRAP_SAMPLES) for label, fit in fits.items()}


def plot_errors(results, target_index=1):
    """Draw one panel per observable and one bar per result, showing estimate - ideal."""
    x = np.arange(len(results))
    _figure, panels = plt.subplots(
        1,
        len(OBSERVABLE_NAMES),
        figsize=(1.2 * len(results) * len(OBSERVABLE_NAMES) + 2.5, 3.4),
        constrained_layout=True,
        squeeze=False,
    )
    for position, (name, panel) in enumerate(zip(OBSERVABLE_NAMES, panels[0])):
        ideal = ideal_observables[target_index][position]
        panel.bar(
            x,
            [out.values[target_index, position] - ideal for out in results.values()],
            0.6,
            yerr=[out.sigma[target_index, position] if out.sigma is not None else 0.0 for out in results.values()],
            capsize=3,
            color=COMPARISON_COLOURS[position],
        )
        panel.axhline(0.0, color="black", linewidth=1)
        panel.set_xticks(x, list(results), rotation=20)
        panel.set_title(name, fontsize="medium")
        panel.spines[["top", "right"]].set_visible(False)

    panels[0][0].set_ylabel("estimate - ideal")
    plt.show()


plot_errors(results)

In [ ]:
fig = plot_zne_comparison(results, target_index=1, pauli_term="ZIZI", ideal=ideal_terms[1][2])
plt.show()

### 4.2 Fitting a subset of the noise factors

The circuits are already executed, so which of them to fit through is free to change. `noise_factors=` selects from the executed noise factors, and `None` fits all of them. A noise factor that was not executed is refused.

In [ ]:
subsets = {"1, 3, 7, 9": None, "1, 3, 7": (1, 3, 7), "1, 3": (1, 3)}

plot_errors(
    {
        label: workflow.estimate(
            ZNEEstimation(degree=1, noise_factors=selection), bootstrap_samples=BOOTSTRAP_SAMPLES
        )
        for label, selection in subsets.items()
    }
)